# PySpark Handling Duplicate columns after join



# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DuplicateCols") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/05/13 23:13:16 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/13 23:13:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-afd996bd-9a16-4d38-be8e-92183ffe0877;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 192ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

# Create 2 dataframe of test data

In [9]:
orders = spark.createDataFrame([
    (1, "Laptop", 101),
    (2, "Phone",  102),
    (3, "Tablet", 101),
], ["id", "name", "customer_id"])

customers = spark.createDataFrame([
    (101, "Alice", "alice@email.com"),
    (102, "Bob",   "bob@email.com"),
], ["id", "name", "email"])

# The Problem First


In [10]:
# Both DataFrames have 'id' and 'name' — this creates duplicates
joined = orders.join(customers, orders.customer_id == customers.id)

joined.printSchema()
# root
#  |-- id: long        ← orders.id
#  |-- name: string    ← orders.name
#  |-- customer_id: long
#  |-- id: long        ← customers.id   (DUPLICATE!)
#  |-- name: string    ← customers.name (DUPLICATE!)
#  |-- email: string

joined.select("id")   # AnalysisException: Ambiguous column name 'id'


root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)



AnalysisException: [AMBIGUOUS_REFERENCE] Reference `id` is ambiguous, could be: [`id`, `id`].

# Fix 1 — Alias DataFrames + Explicit select()

In [4]:
result = (
    orders.alias("o")
    .join(customers.alias("c"), col("o.customer_id") == col("c.id"))
    .select(
        col("o.id").alias("order_id"),
        col("o.name").alias("order_name"),
        col("c.id").alias("customer_id"),
        col("c.name").alias("customer_name"),
        col("c.email"),
    )
)

result.show()


+--------+----------+-----------+-------------+---------------+
|order_id|order_name|customer_id|customer_name|          email|
+--------+----------+-----------+-------------+---------------+
|       1|    Laptop|        101|        Alice|alice@email.com|
|       3|    Tablet|        101|        Alice|alice@email.com|
|       2|     Phone|        102|          Bob|  bob@email.com|
+--------+----------+-----------+-------------+---------------+



# Fix 2 — Drop Duplicate Columns After Join

In [5]:
joined = orders.join(customers, orders.customer_id == customers.id)

# Drop the customer 'id' and 'name' using the DataFrame reference
# (must use DataFrame.col — string "id" would be ambiguous)
result = joined.drop(customers["id"]).drop(customers["name"])

result.show()

+---+------+-----------+---------------+
| id|  name|customer_id|          email|
+---+------+-----------+---------------+
|  1|Laptop|        101|alice@email.com|
|  3|Tablet|        101|alice@email.com|
|  2| Phone|        102|  bob@email.com|
+---+------+-----------+---------------+



# Fix 3 — Join on Shared Key with USING-style (string key)

In [11]:
# When join key has the SAME name in both DataFrames,
# pass it as a string — PySpark keeps only one copy automatically
orders2 = orders.withColumnRenamed("customer_id", "id")  # rename to match

result = orders2.join(customers, on="id")   # single 'id' in output

result.show()

+---+----+---+----+-----+
| id|name| id|name|email|
+---+----+---+----+-----+
+---+----+---+----+-----+



# Fix 4 — Rename Before Joining (Cleanest for Pipelines)

In [12]:
orders_clean = (orders
    .withColumnRenamed("id",   "order_id")
    .withColumnRenamed("name", "order_name")
)

customers_clean = (customers
    .withColumnRenamed("id",   "customer_id")
    .withColumnRenamed("name", "customer_name")
)

result = orders_clean.join(customers_clean, on="customer_id")

result.show()

+-----------+--------+----------+-------------+---------------+
|customer_id|order_id|order_name|customer_name|          email|
+-----------+--------+----------+-------------+---------------+
|        101|       1|    Laptop|        Alice|alice@email.com|
|        101|       3|    Tablet|        Alice|alice@email.com|
|        102|       2|     Phone|          Bob|  bob@email.com|
+-----------+--------+----------+-------------+---------------+

